Step 0: Import Necessary Libraries

In [1]:
import pandas as pd

Step 1: Data Acquisition and Preprocessing

In [2]:
# Import the faculty mesh terms from the Excel file
faculty_mesh_terms_df = pd.read_excel("/Users/sarkisj/Library/CloudStorage/OneDrive-UCIrvine/BioSci Research Development/Faculty-Keyword-Inventory-Project/faculty-mapped-mesh-terms/faculty_unique_mesh_terms.xlsx")

# Create a dictionary with faculty names as keys and their unique MeSH terms as values
faculty_terms = faculty_mesh_terms_df.set_index('Faculty_Full_Name')['Unique_Mesh_Terms'].to_dict()

# Import the funding opportunities mesh terms from the Excel file
funding_terms = [
    "United States",
    "Economic Development",
    "Green Fluorescent Proteins",
    "Universities",
    "Gene Editing",
    "Leadership",
    "Biotechnology",
    "Research",
    "Delivery of Health Care"
]

Step 1a: Test Accessing Dictionary with Faculty Name

In [3]:
# Test accessing the dictionary with a specific faculty name
faculty_name = "Hughes, Christopher"
if faculty_name in faculty_terms:
    print(f"Mesh Terms for {faculty_name}: {faculty_terms[faculty_name]}")
else:
    print(f"Faculty {faculty_name} not found in the dictionary.")

Mesh Terms for Hughes, Christopher: Adipogenesis; Adipose Tissue; Allografts; Alzheimer Disease; Anemia; Angiogenesis; Anxiety; Apolipoprotein E2; Apolipoprotein E4; Arteries; Arteriovenous Malformations; Astrocytes; Basement Membrane; Bayes Theorem; Biomarkers, Tumor; Biomimetics; Blood Vessels; Blood-Brain Barrier; Brain; Breast; Capillaries; Cell Line, Tumor; Cell- and Tissue-Based Therapy; Clustered Regularly Interspaced Short Palindromic Repeats; Coculture Techniques; Colonic Neoplasms; Colorectal Neoplasms; Cytoreduction Surgical Procedures; Diabetes Mellitus; Drug Development; Drug Evaluation; Drug Interactions; Drug Repositioning; Endoglin; Endothelial Cells; Endothelium, Vascular; Epistaxis; Exosomes; Extracellular Matrix; Fascia; Fibroblasts; Fluorouracil; Gene Expression; Glucose; Heart Failure; Huntington Disease; Hyperthermia, Induced; Hyperthermic Intraperitoneal Chemotherapy; Immunotherapy; Immunotherapy, Adoptive; Incidence; Induced Pluripotent Stem Cells; Infant; Insul

Step 2: Creating Function to Calculate Similarity (Set-Based)

In [4]:
# Function to calculate Jaccard similarity between two sets
def jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    if union == 0:
        return 0.0
    return intersection / union

Step 3: Ranking Faculty Members

In [5]:
# Function to predict faculty fit based on MeSH terms
def predict_faculty_fit(faculty_data, funding_terms):
    """
    Predicts faculty fit for a funding opportunity based on MeSH terms.

    Args:
        faculty_data (dict): Dictionary of faculty IDs and their MeSH term lists (sets).
        funding_terms (set): Set of MeSH terms for the funding opportunity.

    Returns:
        list: A list of tuples containing (faculty_id, similarity_score, overlapping_terms)
              sorted in descending order of similarity.
    """
    faculty_scores = []
    for faculty_id, faculty_terms in faculty_data.items():
        overlap = set(faculty_terms).intersection(funding_terms)
        similarity = jaccard_similarity(set(faculty_terms), funding_terms)
        faculty_scores.append((faculty_id, similarity, overlap))

    # Sort by similarity score in descending order
    ranked_faculty = sorted(faculty_scores, key=lambda x: x[1], reverse=True)
    return ranked_faculty

# Convert faculty terms to sets, skipping any null values
faculty_mesh_sets = {}
for k, v in faculty_terms.items():
    if pd.notna(v):  # Check if the value is not NaN
        # Split the string by semicolon and strip whitespace
        terms = set(term.strip() for term in v.split(';'))
        faculty_mesh_sets[k] = terms

ranked_results = predict_faculty_fit(faculty_mesh_sets, set(funding_terms))

print("\nRanked Faculty by Fit:")
print("-" * 80)
print(f"{'Faculty':<30} {'Score':<10} {'Overlapping Terms'}")
print("-" * 80)

for faculty, score, overlap in ranked_results:
    if score > 0:  # Only show faculty with non-zero scores
        print(f"{faculty:<30} {score:.4f}    {', '.join(sorted(overlap))}")


Ranked Faculty by Fit:
--------------------------------------------------------------------------------
Faculty                        Score      Overlapping Terms
--------------------------------------------------------------------------------
Donovan, Peter                 0.0345    Leadership
Fortin, Norbert                0.0270    Leadership
Daley, Monica                  0.0267    Biotechnology, Leadership
LaFerla, Frank                 0.0247    Gene Editing, Leadership
Martiny, Jennifer              0.0244    Biotechnology, Leadership
Schechtman-Drayman, Eitan      0.0222    Leadership
Ribbe, Markus                  0.0213    Biotechnology
Ranz, Jose                     0.0204    Research
Hu, Yilin                      0.0192    Biotechnology
Tinoco, Roberto                0.0174    Delivery of Health Care, Research
Morehouse, Benjamin            0.0141    Delivery of Health Care
McNulty, Reginald              0.0139    Gene Editing
Briscoe, Adriana               0.0135    Gen